# Coarsening from the 700p shots (2026-08-19)

Same pipeline as `shot_images.ipynb`, run on `700p_shots/2026_08_19/`.

What is different from the 126-shot `coarsening_data` set:

| | `coarsening_data` | `700p_shots/2026_08_19` |
|---|---|---|
| shots | 129 | 729 usable (752 files) |
| times | 26 (0-800 ms) | 29 (0-2000 ms) |
| shots per time | ~5 | **25-26** |
| crop box | `[130, 400, 105, 380]` (hard-coded) | measured from the t = 0 mean (below) |

The many-shots-per-time difference is the important one: the per-time PCA is
built from ~25 samples instead of ~5, so PC1 is a far better-determined
estimate of the shared envelope and the self-subtraction bias is ~1/25
instead of ~1/5.

Time map (matches the sequence log in `2026_08_19_coarsening.ipynb`, and
verified against `BEC_TIME_SEQ[6]` in every file):
seq 6 -> 0 ms; 7 -> 800; 9-12 -> 20,30,40,50; 14 -> 32,34,36,38;
15 -> 60,70,80,90,100; 16 -> 25; 18 -> 150,200,250,300;
19 -> 400,500,600,700; 20 -> 1000,1333,1667,2000; 21 -> 45.

In [ ]:
import os
import re
import glob
import sys

import numpy as np
from scipy.ndimage import rotate, gaussian_filter
import h5py
import matplotlib.pyplot as plt
%matplotlib inline

SCRIPT_DIR = os.path.dirname(globals().get("__file__", "")) or os.path.abspath("")
if os.path.basename(SCRIPT_DIR) != "scripts":
    SCRIPT_DIR = "/Users/nolexnol/quantum_sim/scripts"

from quantum_sim.config import dataset
from quantum_sim.io import ShotFile

DATA_DIR = str(dataset("shots700p").path)

CAM = "cam6"
TIME_INDEX = 6        # swept element of BEC_TIME_SEQ (seconds)
CROP_HALF = 130       # half-width of the square cut around the cloud centroid
ROT_ANGLE = 45        # deg; the box sits diamond-wise in the raw frame

# Pixel size. The 08/19 notebook and shot_images cell 3 both use 0.66 um/px.
# Sanity check below: the box measures ~173 px across in BOTH datasets, so a
# 113 um box implies 113/173 = 0.653 um/px. 0.696 would make the box 120 um.
# L scales linearly with this, so switching is a one-line change.
PIXEL_UM = 0.66

NAME_RE = re.compile(
    r"^(?P<seq>\d{4}-\d{2}-\d{2}_\d{4})_bec_box2_(?P<idx>\d+)"
    r"(?:_rep(?P<rep>\d+))?\.h5$"
)


def shot_paths():
    """Every bec_box2 .h5 in DATA_DIR that actually contains camera frames.

    23 of the 752 files (the last rep of each sequence run) were written
    without an /images group -- aborted shots. They are dropped here.
    """
    keep, dead = [], []
    for path in sorted(glob.glob(os.path.join(DATA_DIR, "*bec_box2*.h5"))):
        if not NAME_RE.match(os.path.basename(path)):
            continue
        with h5py.File(path, "r") as f:
            (keep if "images" in f else dead).append(path)
    print(f"{len(keep)} usable shots, {len(dead)} dropped (no /images group)")
    return keep


def shot_time_ms(path):
    """Evolution time of one shot, in ms, straight from its globals."""
    with h5py.File(path, "r") as f:
        return round(float(np.array(f["globals"].attrs["BEC_TIME_SEQ"])[TIME_INDEX]) * 1e3, 3)


def rotate_and_crop(od, angle=ROT_ANGLE):
    """Rotate and crop to the inscribed square so no zero padding remains."""
    rot = rotate(od, angle, reshape=False)
    c0, c1 = rot.shape[0] // 2, rot.shape[1] // 2
    h = int(min(rot.shape) / (2 * np.sqrt(2)))
    return rot[c0 - h:c0 + h, c1 - h:c1 + h]


paths = shot_paths()

## Locate the cloud

`shot_images.ipynb` hard-codes `CROP_BOX = [130, 400, 105, 380]`, which is the
cloud position in the *July* frames. This run is a different day, so the box is
measured here instead: centroid of the top 10% of the blurred t = 0 mean OD
(the recipe from `2026_08_19_coarsening.ipynb`), then a `CROP_HALF`-radius
square around it.

In [ ]:
def mean_od(paths_, crop_box=None):
    return np.mean([ShotFile(p, cam=CAM, crop_box=crop_box).od for p in paths_], axis=0)


# sequence 6 is the t = 0 reference run
t0_paths = [p for p in paths if shot_time_ms(p) == 0.0]
m_full = mean_od(t0_paths)
print(f"t = 0 reference: {len(t0_paths)} shots, full frame {m_full.shape}")

_b = gaussian_filter(m_full, sigma=5)
_thr = np.sort(_b.ravel())[-int(_b.size * 0.10)]
_w = np.where(_b < _thr, 0.0, _b)
_sr, _sc = _w.sum(axis=1), _w.sum(axis=0)
CTR_ROW = int((np.arange(_w.shape[0]) * _sr).sum() / _sr.sum())
CTR_COL = int((np.arange(_w.shape[1]) * _sc).sum() / _sc.sum())

CROP_BOX = [CTR_ROW - CROP_HALF, CTR_ROW + CROP_HALF,
            CTR_COL - CROP_HALF, CTR_COL + CROP_HALF]
print(f"centroid (row, col) = ({CTR_ROW}, {CTR_COL})")
print(f"CROP_BOX = {CROP_BOX}")

# --- pixel-size sanity check: how wide is the box, in px? ---
_crop = m_full[CROP_BOX[0]:CROP_BOX[1], CROP_BOX[2]:CROP_BOX[3]]
_rot = rotate(_crop, ROT_ANGLE, reshape=False)
_widths = []
for _ax in (0, 1):
    _pr = _rot.sum(axis=_ax)
    _pr = _pr - np.median(np.r_[_pr[:10], _pr[-10:]])
    _i = np.where(_pr > _pr.max() * 0.35)[0]
    _widths.append(_i.max() - _i.min() + 1)
_wpx = np.mean(_widths)
print(f"box width = {_wpx:.0f} px  ->  {_wpx * PIXEL_UM:.1f} um at {PIXEL_UM} um/px"
      f"   (a 113 um box would need {113 / _wpx:.3f} um/px)")

# --- what the analysis window looks like ---
fig, axes = plt.subplots(1, 4, figsize=(16, 4.2))
axes[0].imshow(m_full)
axes[0].plot(CTR_COL, CTR_ROW, "r+", ms=14)
axes[0].set_title("t = 0 mean OD, full frame")
axes[1].imshow(_crop)
axes[1].set_title(f"{_crop.shape[0]}x{_crop.shape[1]} crop at centroid")
_h = int(min(_crop.shape) / (2 * np.sqrt(2)))
axes[2].imshow(_rot)
axes[2].add_patch(plt.Rectangle((CROP_HALF - _h, CROP_HALF - _h), 2 * _h, 2 * _h,
                                fill=False, ec="r", lw=1.5))
axes[2].set_title(f"rotated {ROT_ANGLE} deg (red = kept)")
axes[3].imshow(rotate_and_crop(_crop))
axes[3].set_title(f"rotate_and_crop -> {2 * _h}x{2 * _h}")
for _a in axes:
    _a.set_xticks([])
    _a.set_yticks([])
fig.tight_layout()
plt.show()

Note the inscribed square (182 px) is a little **wider** than the box
itself (~173 px), so a few px of outside-box wall sit in the analysis window --
exactly the same situation as in `shot_images.ipynb`. It is left in for
comparability; the last cell checks that it does not matter.

In [ ]:
# Build the image stack: one rotated/cropped OD per shot, tagged by time.
t_shots, _stack = [], []
for i, p in enumerate(paths):
    if i % 100 == 0:
        print(f"  {i}/{len(paths)}", flush=True)
    sf = ShotFile(p, cam=CAM, crop_box=CROP_BOX)
    t_shots.append(shot_time_ms(p))
    _stack.append(rotate_and_crop(sf.od))

t_shots = np.array(t_shots)
stack = np.stack(_stack)                 # [shot, x, y]
times = sorted(set(t_shots.tolist()))
del _stack

print(f"\nimage stack: {stack.shape}")
print(f"{len(times)} times, {t_shots.size} shots")
for _t in times:
    print(f"  t = {_t:8.1f} ms : {int((t_shots == _t).sum())} shots")

In [ ]:
# Same three functions as shot_images.ipynb, verbatim, so the two datasets
# are analysed identically.
def pca_svd(images):
    """Return normalized S^2 and spatial PCA vectors for images [shot, x, y]."""
    images = np.asarray(images, dtype=float)
    if images.ndim != 3:
        raise ValueError("images must have shape [shot, x, y]")
    pixels_by_shot = images.reshape(images.shape[0], -1).T
    vectors, singular_values, _ = np.linalg.svd(pixels_by_shot, full_matrices=False)
    eigenvalues = singular_values**2
    total = eigenvalues.sum()
    if total == 0:
        raise ValueError("images must have non-zero variance")
    eigenvalues /= total
    return eigenvalues, vectors.reshape(*images.shape[1:], -1)


def remove_pca_components(image, eigenvectors, n_components):
    """Return (cleaned, removed) after projecting image onto leading components."""
    image = np.asarray(image, dtype=float)
    eigenvectors = np.asarray(eigenvectors, dtype=float)
    if image.ndim != 2 or eigenvectors.shape[:2] != image.shape:
        raise ValueError("image and eigenvectors have incompatible shapes")
    if not 0 <= n_components <= eigenvectors.shape[-1]:
        raise ValueError("n_components is outside the available basis")
    basis = eigenvectors[..., :n_components].reshape(image.size, n_components)
    removed = (basis @ (basis.T @ image.ravel())).reshape(image.shape)
    return image - removed, removed


def _domain_length_raw(od):
    """L = 2 pi <S> / <k S> in px, only the uniform mode dropped."""
    S = np.abs(np.fft.fft2(od))**2
    S[0, 0] = 0.0
    ny, nx = od.shape
    ky = 2 * np.pi * np.fft.fftfreq(ny)
    kx = 2 * np.pi * np.fft.fftfreq(nx)
    KX, KY = np.meshgrid(kx, ky, indexing="xy")
    return 2 * np.pi * S.sum() / (np.sqrt(KX**2 + KY**2) * S).sum()


# Per-time PCA: each time's shots get their OWN basis, so the envelope is
# tracked as it drifts over the 2 s of evolution.  ~25 samples per fit here.
pca_by_time = {_t: pca_svd(stack[t_shots == _t]) for _t in times}
print("per-time PCA done;  PC1 share of total variance:")
for _t in times:
    print(f"  t = {_t:8.1f} ms : {pca_by_time[_t][0][0]:.4f}")

In [ ]:
# Eigenvalue spectra, one panel per time (log-lin), as in shot_images cell 5.
N_EIG_SHOW = 12
ncols = 6
nrows = int(np.ceil(len(times) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(2.7 * ncols, 2.4 * nrows), sharey=True)
for ax, _t in zip(axes.ravel(), times):
    evals_t, _ = pca_by_time[_t]
    n = min(N_EIG_SHOW, len(evals_t))
    ax.semilogy(np.arange(1, n + 1), evals_t[:n], "o-", ms=3, color="C0")
    ax.set_title(f"t = {_t:g} ms  ({len(evals_t)} shots)", fontsize=9)
    ax.tick_params(labelsize=8)
    ax.grid(alpha=0.25, which="both")
for ax in axes.ravel()[len(times):]:
    ax.axis("off")
fig.supxlabel("component #")
fig.supylabel("eigenvalue (fraction of total)")
fig.suptitle(f"log-lin pca_svd eigenvalues, first {N_EIG_SHOW} of each per-time PCA")
fig.tight_layout()
plt.show()

In [ ]:
# The k-th eigenvector of each per-time PCA as an image, PC1..PC4.
ncols = 6
nrows = int(np.ceil(len(times) / ncols))
for k in range(4):
    fig, axes = plt.subplots(nrows, ncols, figsize=(2.7 * ncols, 2.6 * nrows))
    for ax, _t in zip(axes.ravel(), times):
        evals_t, evecs_t = pca_by_time[_t]
        ax.set_xticks([])
        ax.set_yticks([])
        if k >= evecs_t.shape[-1]:
            ax.axis("off")
            continue
        comp = evecs_t[..., k]
        lim = np.abs(comp).max()
        ax.imshow(comp, cmap="RdBu_r", vmin=-lim, vmax=lim)
        ax.set_title(f"t = {_t:g} ms  ({evals_t[k]:.1%})", fontsize=9)
    for ax in axes.ravel()[len(times):]:
        ax.axis("off")
    fig.suptitle(f"PC{k + 1} of each per-time PCA (percent = its eigenvalue)")
    fig.tight_layout()
    plt.show()

In [ ]:
# removes the first principal component of EACH TIME'S OWN PCA, NO window.
# L = 2 pi <S> / <k S> straight on the cleaned shot.
L_shots = np.empty(len(stack))
for _t in times:
    _m = t_shots == _t
    _, _pcs_t = pca_by_time[_t]
    L_shots[_m] = [_domain_length_raw(remove_pca_components(im, _pcs_t, 1)[0])
                   for im in stack[_m]]
L_shots *= PIXEL_UM

_tm = np.array(times, float)
L_mean = np.array([L_shots[t_shots == t].mean() for t in times])
L_sem = np.array([L_shots[t_shots == t].std(ddof=1)
                  / np.sqrt((t_shots == t).sum()) for t in times])

print(f"{'t (ms)':>9} {'n':>4} {'L (um)':>9} {'sem':>7}")
for i, t in enumerate(times):
    print(f"{t:9.1f} {int((t_shots == t).sum()):4d} {L_mean[i]:9.2f} {L_sem[i]:7.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(t_shots, L_shots, "o", ms=3, color="C0", alpha=0.25,
        label="individual shots")
ax.errorbar(_tm, L_mean, yerr=L_sem, fmt="o", ms=6, color="C0",
            capsize=3, label="mean of repeats", zorder=3)
ax.set_xlabel("evolution time  t  (ms)")
ax.set_ylabel(r"domain size  L  ($\mu$m)")
ax.set_title("700p shots: domain size vs evolution time\n(per-time first PC removed, no window)")
ax.legend(frameon=False)
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

In [ ]:
# log-log, with free power-law fits over a few windows and a t^(1/3) guide.
fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(_tm[_tm > 0], L_mean[_tm > 0], yerr=L_sem[_tm > 0], fmt="o", ms=6,
            color="C0", capsize=3, label="mean of repeats", zorder=3)

print("free power-law fits  L ~ t^n :")
for (lo, hi), c in zip([(20, 100), (30, 300), (20, 800)], ["C1", "C2", "C3"]):
    s = (_tm >= lo) & (_tm <= hi)
    n, logA = np.polyfit(np.log(_tm[s]), np.log(L_mean[s]), 1)
    print(f"  {lo:5.0f}-{hi:5.0f} ms  (n_pts={s.sum():2d}) :  n = {n:.3f}")
    ax.plot(_tm[s], np.exp(logA) * _tm[s]**n, "-", color=c, lw=1.6,
            label=f"{lo}-{hi} ms: $t^{{{n:.2f}}}$")

_s13 = (_tm >= 20) & (_tm <= 300)
_A = np.exp(np.mean(np.log(L_mean[_s13]) - np.log(_tm[_s13]) / 3))
ax.plot(_tm[_tm > 0], _A * _tm[_tm > 0]**(1 / 3), "k--", lw=1.2, label=r"$t^{1/3}$ guide")

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("evolution time  t  (ms)")
ax.set_ylabel(r"domain size  L  ($\mu$m)")
ax.set_title("700p shots: log-log growth")
ax.legend(frameon=False, fontsize=9)
ax.grid(alpha=0.25, which="both")
fig.tight_layout()
plt.show()

## Robustness

Two things were chosen by hand above: the analysis window (the inscribed square
is slightly wider than the box) and removing exactly one component. This sweeps
both. If the numbers move a lot here, the growth exponent is an artefact of
those choices rather than a measurement.

In [ ]:
rot_full = np.stack([rotate(ShotFile(p, cam=CAM, crop_box=CROP_BOX).od,
                             ROT_ANGLE, reshape=False) for p in paths])

def sweep(half, n_pc):
    c = rot_full.shape[1] // 2
    st = rot_full[:, c - half:c + half, c - half:c + half]
    L = np.empty(len(st))
    for t in times:
        m = t_shots == t
        _, pcs = pca_svd(st[m])
        L[m] = [_domain_length_raw(remove_pca_components(im, pcs, n_pc)[0]) for im in st[m]]
    L *= PIXEL_UM
    return np.array([L[t_shots == t].mean() for t in times])

print(f"{'window':>10} {'nPC':>4} | {'L(0)':>6} {'L(40)':>6} {'L(300)':>7} {'L(2000)':>8} |"
      f" {'n(20-100)':>10} {'n(30-300)':>10}")
for half in (91, 84, 75):
    for n_pc in (1, 2, 3):
        m = sweep(half, n_pc)
        e1 = np.polyfit(np.log(_tm[(_tm >= 20) & (_tm <= 100)]),
                        np.log(m[(_tm >= 20) & (_tm <= 100)]), 1)[0]
        e2 = np.polyfit(np.log(_tm[(_tm >= 30) & (_tm <= 300)]),
                        np.log(m[(_tm >= 30) & (_tm <= 300)]), 1)[0]
        idx = lambda v: times.index(v)
        print(f"{2 * half:10d} {n_pc:4d} | {m[0]:6.2f} {m[idx(40.0)]:6.2f} "
              f"{m[idx(300.0)]:7.2f} {m[idx(2000.0)]:8.2f} | {e1:10.3f} {e2:10.3f}")

## Things to look at

- **t = 45 ms sits well below its neighbours** (~4.9 um against ~7.4 um at both
  40 and 50 ms). That is sequence 21, taken on its own after the rest; worth
  checking whether that run had different loading before trusting the point.
- **L turns over after ~300-800 ms** and falls by 2000 ms. The box is ~113 um
  and L only reaches ~14 um, so this is not domains hitting the wall -- more
  likely contrast loss as atoms are lost, which drags L down through the
  high-k noise floor.
- **L(t = 0) = 2.5 um** is the noise floor (~4 px), which is the sanity check
  that PC1 removal is taking the envelope out properly.

---

## Tukey window

Everything above feeds the raw cropped residual straight into `fft2`. The
analysis window is a hard square, so the discontinuity at its edge leaks power
across all k and biases `L = 2 pi <S> / <k S>`. A Tukey (tapered-cosine) window
rolls the edges to zero over a fraction `alpha` of the width; `alpha = 0` is
the plain rectangle (i.e. exactly what the cells above do) and `alpha = 1` is a
full Hann.

Order matters, so to be explicit -- for each shot:

1. `rotate_and_crop` (already done, `stack`)
2. remove PC1 of **that time's own** PCA
3. subtract the residual mean
4. multiply by the 2D Tukey window
5. `fft2` -> `S(k)` -> `L`

Step 3 is the easy one to drop. Windowing a field with a non-zero mean spreads
its DC component into a low-k skirt shaped like the window's own transform, and
zeroing `S[0, 0]` afterwards does *not* remove that skirt. In practice PC1
removal already leaves the residual nearly centred here (median
`|mean| / rms = 0.02`), so it changes `L` by <0.2% -- but it costs nothing and
is correct.

In [ ]:
from scipy.signal.windows import tukey

WIN_ALPHA = 0.5      # 0 = no window (rectangle), 1 = full Hann


def _tukey2d(shape, alpha=WIN_ALPHA):
    """Separable 2D Tukey window."""
    return np.outer(tukey(shape[0], alpha), tukey(shape[1], alpha))


def _domain_length_windowed(od, window):
    """L = 2 pi <S> / <k S>, on a de-meaned, apodized field."""
    fluct = od - od.mean()
    if window is not None:
        fluct = fluct * window
    return _domain_length_raw(fluct)


_win = _tukey2d(stack.shape[1:], WIN_ALPHA)

L_shots_win = np.empty(len(stack))
for _t in times:
    _m = t_shots == _t
    _, _pcs_t = pca_by_time[_t]
    L_shots_win[_m] = [_domain_length_windowed(remove_pca_components(im, _pcs_t, 1)[0], _win)
                       for im in stack[_m]]
L_shots_win *= PIXEL_UM

L_mean_win = np.array([L_shots_win[t_shots == t].mean() for t in times])
L_sem_win = np.array([L_shots_win[t_shots == t].std(ddof=1)
                      / np.sqrt((t_shots == t).sum()) for t in times])

print(f"Tukey alpha = {WIN_ALPHA}")
print(f"{'t (ms)':>9} {'n':>4} {'L win':>8} {'sem':>7} {'L no-win':>9} {'change':>8}")
for i, t in enumerate(times):
    d = 100 * (L_mean_win[i] - L_mean[i]) / L_mean[i]
    print(f"{t:9.1f} {int((t_shots == t).sum()):4d} {L_mean_win[i]:8.2f} {L_sem_win[i]:7.3f}"
          f" {L_mean[i]:9.2f} {d:7.1f}%")

In [ ]:
# the same graph as above, now with the Tukey window (first PC still removed)
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(t_shots, L_shots_win, "o", ms=3, color="C0", alpha=0.25,
        label="individual shots")
ax.errorbar(_tm, L_mean_win, yerr=L_sem_win, fmt="o", ms=6, color="C0",
            capsize=3, label=f"mean of repeats (Tukey $\\alpha$={WIN_ALPHA})", zorder=3)
ax.errorbar(_tm, L_mean, yerr=L_sem, fmt="s", ms=5, color="C3", alpha=0.75,
            capsize=3, label="mean, no window", zorder=2)
ax.set_xlabel("evolution time  t  (ms)")
ax.set_ylabel(r"domain size  L  ($\mu$m)")
ax.set_title(f"700p shots: domain size vs evolution time\n"
             f"(per-time first PC removed, Tukey $\\alpha$ = {WIN_ALPHA})")
ax.legend(frameon=False)
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

In [ ]:
# log-log companion, with the free power-law fits
fig, ax = plt.subplots(figsize=(8, 5))
_p = _tm > 0
ax.errorbar(_tm[_p], L_mean_win[_p], yerr=L_sem_win[_p], fmt="o", ms=6, color="C0",
            capsize=3, label=f"Tukey $\\alpha$={WIN_ALPHA}", zorder=3)
ax.errorbar(_tm[_p], L_mean[_p], yerr=L_sem[_p], fmt="s", ms=5, color="C3", alpha=0.6,
            capsize=3, label="no window", zorder=2)

print(f"free power-law fits  L ~ t^n   (Tukey alpha = {WIN_ALPHA}):")
for (lo, hi), c in zip([(20, 100), (30, 300), (20, 800)], ["C1", "C2", "C4"]):
    s = (_tm >= lo) & (_tm <= hi)
    n, logA = np.polyfit(np.log(_tm[s]), np.log(L_mean_win[s]), 1)
    n0 = np.polyfit(np.log(_tm[s]), np.log(L_mean[s]), 1)[0]
    print(f"  {lo:5.0f}-{hi:5.0f} ms :  n = {n:.3f}   (no window: {n0:.3f})")
    ax.plot(_tm[s], np.exp(logA) * _tm[s]**n, "-", color=c, lw=1.6,
            label=f"{lo}-{hi} ms: $t^{{{n:.2f}}}$")

_s13 = (_tm >= 20) & (_tm <= 300)
_A = np.exp(np.mean(np.log(L_mean_win[_s13]) - np.log(_tm[_s13]) / 3))
ax.plot(_tm[_p], _A * _tm[_p]**(1 / 3), "k--", lw=1.2, label=r"$t^{1/3}$ guide")

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("evolution time  t  (ms)")
ax.set_ylabel(r"domain size  L  ($\mu$m)")
ax.set_title(f"700p shots: log-log growth (Tukey $\\alpha$ = {WIN_ALPHA})")
ax.legend(frameon=False, fontsize=9)
ax.grid(alpha=0.25, which="both")
fig.tight_layout()
plt.show()

In [ ]:
# How much does the taper fraction actually matter?
# alpha = 0 must reproduce the unwindowed numbers exactly -- that is the check
# that the window is being applied where it is meant to be.
_alphas = [0.0, 0.1, 0.25, 0.5, 0.75, 1.0]
print(f"{'alpha':>7} | {'L(0)':>6} {'L(40)':>6} {'L(100)':>7} {'L(300)':>7} {'L(2000)':>8} |"
      f" {'n(20-100)':>10} {'n(30-300)':>10}")
_idx = lambda v: times.index(v)
for _a in _alphas:
    _w = _tukey2d(stack.shape[1:], _a)
    _L = np.empty(len(stack))
    for _t in times:
        _m = t_shots == _t
        _, _pcs_t = pca_by_time[_t]
        _L[_m] = [_domain_length_windowed(remove_pca_components(im, _pcs_t, 1)[0], _w)
                  for im in stack[_m]]
    _L *= PIXEL_UM
    _m_ = np.array([_L[t_shots == t].mean() for t in times])
    _e1 = np.polyfit(np.log(_tm[(_tm >= 20) & (_tm <= 100)]),
                     np.log(_m_[(_tm >= 20) & (_tm <= 100)]), 1)[0]
    _e2 = np.polyfit(np.log(_tm[(_tm >= 30) & (_tm <= 300)]),
                     np.log(_m_[(_tm >= 30) & (_tm <= 300)]), 1)[0]
    print(f"{_a:7.2f} | {_m_[0]:6.2f} {_m_[_idx(40.0)]:6.2f} {_m_[_idx(100.0)]:7.2f} "
          f"{_m_[_idx(300.0)]:7.2f} {_m_[_idx(2000.0)]:8.2f} | {_e1:10.3f} {_e2:10.3f}")

### What the window does here

The taper raises `L` at every evolved time -- ~20% at t = 40 ms for
`alpha = 0.5` -- while leaving `L(t = 0) = 2.5 um` untouched, since that point
is already at the noise floor and has no edge structure to leak.

It does **not** leave the exponent alone: over 20-100 ms it goes 0.530 (no
window) -> 0.591 at `alpha = 0.5` -> 0.604 at `alpha = 1`. So unlike the crop
width and the number of PCs removed -- both of which moved the exponent by
<0.06 in the sweep above -- the apodization is a real lever on the answer, and
whichever `alpha` you settle on has to be quoted with the result.

---

## 2026-09-11 shots: cropped OD only (no PC removed, no window)

New run (`data/raw/2026-09-11`, dataset `sep11`): 692 usable shots over 24 hold
times from 0 to 2000 ms, 26-30 shots per time. It is framed exactly like the 700p
set above (centroid of the t = 0 mean, `CROP_HALF` square, `rotate_and_crop`),
but `L = 2 pi <S> / <k S>` goes straight onto the cropped OD: **no PC is
removed and no window is applied**.

Because nothing is subtracted, the box envelope and walls add to `S(k)`, so
`L(t = 0)` sits near 9 um instead of the ~2.5 um noise floor you get after PC1
removal. Compare these numbers with other no-PC results only. The 700p set is
drawn in grey for reference, run through the same estimator.

In [ ]:
# 2026-09-11 run: domain size vs evolution time, NO PC removed and NO window.
# Same crop as the 700p analysis above: centroid of the blurred t = 0 mean,
# CROP_HALF square around it, then rotate_and_crop. L = 2 pi <S> / <k S> goes
# straight onto that cropped OD, with only the k = 0 mode dropped.
SEP11_DIR = str(dataset("sep11").path)

sep11_paths = []
for _p in sorted(glob.glob(os.path.join(SEP11_DIR, "*bec_box2*.h5"))):
    if not NAME_RE.match(os.path.basename(_p)):
        continue
    with h5py.File(_p, "r") as _f:
        if "images" in _f:                 # skip aborted shots (no frames)
            sep11_paths.append(_p)
print(f"2026-09-11: {len(sep11_paths)} usable shots")

# locate the cloud from this run's own t = 0 shots
_m11 = mean_od([p for p in sep11_paths if shot_time_ms(p) == 0.0])
_b = gaussian_filter(_m11, sigma=5)
_w = np.where(_b < np.sort(_b.ravel())[-int(_b.size * 0.10)], 0.0, _b)
_sr, _sc = _w.sum(axis=1), _w.sum(axis=0)
_r = int((np.arange(_w.shape[0]) * _sr).sum() / _sr.sum())
_c = int((np.arange(_w.shape[1]) * _sc).sum() / _sc.sum())
CROP_BOX_11 = [_r - CROP_HALF, _r + CROP_HALF, _c - CROP_HALF, _c + CROP_HALF]
print(f"centroid (row, col) = ({_r}, {_c})  ->  CROP_BOX_11 = {CROP_BOX_11}")

t_shots_11 = np.array([shot_time_ms(p) for p in sep11_paths])
stack_11 = np.stack([rotate_and_crop(ShotFile(p, cam=CAM, crop_box=CROP_BOX_11).od)
                     for p in sep11_paths])
times_11 = sorted(set(t_shots_11.tolist()))

L_raw_11 = np.array([_domain_length_raw(im) for im in stack_11]) * PIXEL_UM
_tm11 = np.array(times_11, float)
L_mean_11 = np.array([L_raw_11[t_shots_11 == t].mean() for t in times_11])
L_sem_11 = np.array([L_raw_11[t_shots_11 == t].std(ddof=1)
                     / np.sqrt((t_shots_11 == t).sum()) for t in times_11])

# the 700p stack through the identical no-PC / no-window estimator, for reference
L_raw_700 = np.array([_domain_length_raw(im) for im in stack]) * PIXEL_UM
L_mean_700 = np.array([L_raw_700[t_shots == t].mean() for t in times])

print(f"{'t (ms)':>9} {'n':>4} {'L (um)':>9} {'sem':>7}")
for i, t in enumerate(times_11):
    print(f"{t:9.1f} {int((t_shots_11 == t).sum()):4d} {L_mean_11[i]:9.2f} {L_sem_11[i]:7.3f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(t_shots_11, L_raw_11, "o", ms=3, color="C0", alpha=0.25,
        label="2026-09-11, individual shots")
ax.errorbar(_tm11, L_mean_11, yerr=L_sem_11, fmt="o", ms=6, color="C0",
            capsize=3, label="2026-09-11, mean of repeats", zorder=3)
ax.plot(times, L_mean_700, "s--", ms=4, lw=1, color="0.55",
        label="700p (2026-08-19), mean, same estimator")
ax.set_xlabel("evolution time  t  (ms)")
ax.set_ylabel(r"domain size  L  ($\mu$m)")
ax.set_title("2026-09-11 shots: domain size vs evolution time\n"
             "(cropped OD only: no PC removed, no window)")
ax.legend(frameon=False)
ax.grid(alpha=0.25)
fig.tight_layout()
SEP11_FIG = dataset("sep11").figure("domain_size_vs_time_no_pc.png")
fig.savefig(SEP11_FIG, dpi=150)
print(f"saved {SEP11_FIG}")
plt.show()